# 🧠 Human Performance Intelligence System
## Notebook 04 — Module 3: Burnout Risk Scoring
### *Building a composite burnout index and predicting who's at risk*

**Techniques:** Composite Scoring · Logistic Regression · LightGBM · SHAP · Risk Segmentation

> 💡 **What is burnout in a data science context?**
>
> Burnout is a state of chronic stress that leads to physical and emotional exhaustion,
> cynicism, and feelings of ineffectiveness. For a data scientist, it's both a 
> *measurement challenge* (how do you quantify it?) and a *prediction challenge*
> (can you spot it before it happens?).
>
> This module solves both:
>
> **Step 1 — Measurement:** Build a principled 0–100 Burnout Risk Index from multiple 
> behavioral signals (sleep, stress, focus, mood, caffeine). This is called a 
> **composite score** — a weighted combination of indicators, similar to credit scores 
> or BMI.
>
> **Step 2 — Prediction:** Train a classifier to predict which students will enter 
> high-burnout territory in the coming week, *before it happens*, using the previous 
> week's signals.
>
> **Step 3 — Segmentation:** Profile the different burnout risk archetypes and 
> quantify how much each behavioral factor contributes.

**Why this impresses recruiters:**
Burnout risk scoring is directly applicable to HR tech, wellness apps, edtech, and 
enterprise productivity platforms — fast-growing sectors actively hiring data scientists.


## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (roc_auc_score, classification_report,
                              confusion_matrix, f1_score,
                              RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import lightgbm as lgb

plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#FAFAFA',
    'axes.grid':        True,
    'grid.color':       '#E8E8E8',
    'grid.linewidth':   0.8,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.spines.left':   False,
    'axes.spines.bottom': False,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

BLUE   = '#3B8BD4'
RED    = '#E24B4A'
GREEN  = '#3BAD72'
GOLD   = '#F5A623'
PURPLE = '#9B59B6'
ORANGE = '#E67E22'

RISK_COLORS = {
    'Low':     '#3BAD72',
    'Moderate':'#F5A623',
    'High':    '#E24B4A',
    'Critical':'#8B0000',
}

print("✓ Libraries loaded")
import lightgbm; print(f"  LightGBM : {lightgbm.__version__}")
import shap;     print(f"  SHAP     : {shap.__version__}")


ModuleNotFoundError: No module named 'lightgbm'

## 2. Load Data & Build Weekly Feature Matrix

> 💡 **Why weekly aggregation for burnout?**
>
> Burnout is not a single-day event — it's a pattern that accumulates over days.
> A single bad night of sleep doesn't cause burnout. A week of poor sleep + 
> high stress + no social interaction does.
>
> Aggregating to weekly level:
> - Smooths out day-to-day noise (one bad day vs a pattern)
> - Aligns with academic/work rhythms (deadlines, meetings)
> - Gives us enough signal density without losing temporal resolution


In [ ]:
# ── Load clean StudentLife data ─────────────────────────────────────────────
s = pd.read_csv('../data/processed/studentlife_clean.csv', parse_dates=['date'])

print(f"Loaded: {s.shape[0]:,} rows × {s.shape[1]} columns")
print(f"Students: {s['student_id'].nunique():,}  |  Weeks: {s['week_num'].nunique()}")
print(f"Date range: {s['date'].min().date()} → {s['date'].max().date()}")


In [ ]:
# ── Aggregate to weekly level ────────────────────────────────────────────────
weekly = s.groupby(['student_id', 'week_num']).agg(
    avg_sleep          = ('sleep_hours',        'mean'),
    avg_stress         = ('stress_level',       'mean'),
    std_stress         = ('stress_level',       'std'),
    avg_focus          = ('focus_minutes',      'mean'),
    avg_mood           = ('mood_score',         'mean'),
    avg_caffeine       = ('caffeine_drinks',    'mean'),
    avg_activity       = ('activity_steps',     'mean'),
    avg_social         = ('social_hours',       'mean'),
    avg_sleep_debt     = ('sleep_debt',         'mean'),
    avg_stress_mood    = ('stress_mood_ratio',  'mean'),
    avg_productivity   = ('productivity_index', 'mean'),
    avg_goal_rate      = ('goal_completion_rate','mean'),
    avg_burnout        = ('burnout_score',      'mean'),
    is_deadline_week   = ('is_deadline_week',   'max'),
    days_logged        = ('date',               'count'),
).reset_index()

weekly['std_stress'] = weekly['std_stress'].fillna(0)

print(f"Weekly dataset: {weekly.shape[0]:,} rows × {weekly.shape[1]} columns")
print(f"\nBurnout score distribution:")
print(weekly['avg_burnout'].describe().round(3))


## 3. Build the Burnout Risk Index (BRI)

> 💡 **What is a composite score?**
>
> A composite score combines multiple indicators into one number using a 
> weighted formula. Famous examples: credit scores, BMI, Human Development Index.
>
> Our Burnout Risk Index (BRI) is scaled 0–100 where:
> - 0–25  = Low risk (green zone)
> - 26–50 = Moderate risk (amber zone)
> - 51–75 = High risk (red zone)
> - 76–100 = Critical risk (danger zone)
>
> **Weighting rationale (research-backed):**
> Sleep deprivation accounts for ~35% of burnout risk — it's the single 
> most documented physiological burnout predictor in occupational health research.
> Chronic stress contributes ~30%. Low engagement (focus + mood) ~20%.
> Lifestyle factors (caffeine dependency, social isolation) ~15%.


In [ ]:
scaler_norm = MinMaxScaler()

risk_components      = ['avg_stress', 'avg_sleep_debt', 'avg_caffeine', 'avg_stress_mood']
protective_components = ['avg_sleep', 'avg_focus', 'avg_mood', 'avg_social']

all_components = risk_components + protective_components
weekly_norm = pd.DataFrame(
    scaler_norm.fit_transform(weekly[all_components]),
    columns=all_components
)

for col in protective_components:
    weekly_norm[col] = 1 - weekly_norm[col]

WEIGHTS = {
    'avg_sleep_debt':   0.25,
    'avg_sleep':        0.10,
    'avg_stress':       0.20,
    'avg_stress_mood':  0.10,
    'avg_focus':        0.12,
    'avg_mood':         0.08,
    'avg_caffeine':     0.08,
    'avg_social':       0.07,
}

assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, "Weights must sum to 1.0"

weekly['BRI_raw'] = sum(
    weekly_norm[col] * weight
    for col, weight in WEIGHTS.items()
)
weekly['BRI'] = (weekly['BRI_raw'] * 100).clip(0, 100).round(1)

def assign_risk_tier(bri):
    if bri <= 25:   return 'Low'
    elif bri <= 50: return 'Moderate'
    elif bri <= 75: return 'High'
    else:           return 'Critical'

weekly['risk_tier'] = weekly['BRI'].apply(assign_risk_tier)
weekly['risk_tier'] = pd.Categorical(
    weekly['risk_tier'],
    categories=['Low', 'Moderate', 'High', 'Critical'],
    ordered=True
)

print("Burnout Risk Index (BRI) — built ✓")
print(f"\nBRI distribution (0–100 scale):")
print(weekly['BRI'].describe().round(1))
print(f"\nRisk tier counts:")
print(weekly['risk_tier'].value_counts().sort_index())


## 4. Visualize the Burnout Risk Index

Let's validate the BRI makes sense by examining:
1. Its overall distribution
2. Whether it stratifies goal completion rate (does higher BRI → lower productivity?)
3. How it varies across the semester (deadline weeks should spike)


In [ ]:
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(weekly['BRI'], bins=40, color=BLUE, alpha=0.8, edgecolor='white', linewidth=0.5)
ax1.axvspan(0,  25,  alpha=0.12, color=GREEN,     label='Low (0–25)')
ax1.axvspan(25, 50,  alpha=0.12, color=GOLD,      label='Moderate (26–50)')
ax1.axvspan(50, 75,  alpha=0.12, color=RED,       label='High (51–75)')
ax1.axvspan(75, 100, alpha=0.12, color='darkred', label='Critical (76–100)')
ax1.set_title('BRI Distribution')
ax1.set_xlabel('Burnout Risk Index (0–100)')
ax1.set_ylabel('Count')
ax1.legend(fontsize=8, loc='upper right')

ax2 = fig.add_subplot(gs[0, 1])
weekly['bri_bin'] = pd.cut(weekly['BRI'], bins=range(0, 101, 10))
bri_goal = weekly.groupby('bri_bin', observed=True)['avg_goal_rate'].mean() * 100
colors_bar = [GREEN if i < 3 else GOLD if i < 6 else RED if i < 8 else '#8B0000'
              for i in range(len(bri_goal))]
ax2.bar(range(len(bri_goal)), bri_goal.values, color=colors_bar, edgecolor='white', linewidth=0.5)
ax2.set_xticks(range(len(bri_goal)))
ax2.set_xticklabels([str(b) for b in bri_goal.index], rotation=45, fontsize=8)
ax2.set_title('BRI vs Goal Completion Rate\n(validates BRI is meaningful)')
ax2.set_xlabel('BRI bucket')
ax2.set_ylabel('Avg Goal Completion (%)')

ax3 = fig.add_subplot(gs[0, 2])
bri_week = weekly.groupby('week_num')['BRI'].mean()
ax3.plot(bri_week.index, bri_week.values, color=BLUE, linewidth=2.5, marker='o', markersize=5, zorder=3)
ax3.fill_between(bri_week.index, bri_week.values, alpha=0.15, color=BLUE)
deadline_weeks = weekly[weekly['is_deadline_week']==1]['week_num'].unique()
for dw in deadline_weeks:
    if dw in bri_week.index:
        ax3.axvline(dw, color=RED, linestyle='--', linewidth=1, alpha=0.5)
ax3.axhline(50, color=RED, linestyle=':', linewidth=1.5, alpha=0.7, label='High risk threshold (50)')
ax3.set_title('Average BRI Across Semester\n(dashed red = deadline weeks)')
ax3.set_xlabel('Week Number')
ax3.set_ylabel('Average BRI')
ax3.legend(fontsize=9)

ax4 = fig.add_subplot(gs[1, :2])
tier_weekly = weekly.groupby(['week_num', 'risk_tier']).size().unstack(fill_value=0)
tier_weekly_pct = tier_weekly.div(tier_weekly.sum(axis=1), axis=0) * 100
tier_colors = [GREEN, GOLD, RED, '#8B0000']
bottom = np.zeros(len(tier_weekly_pct))
for tier, color in zip(['Low', 'Moderate', 'High', 'Critical'], tier_colors):
    if tier in tier_weekly_pct.columns:
        ax4.bar(tier_weekly_pct.index, tier_weekly_pct[tier],
                bottom=bottom, color=color, label=tier, edgecolor='white', linewidth=0.3)
        bottom += tier_weekly_pct[tier].values
ax4.set_title('Risk Tier Composition by Week (% of students)')
ax4.set_xlabel('Week Number')
ax4.set_ylabel('% of Students')
ax4.legend(loc='upper left', fontsize=9)

ax5 = fig.add_subplot(gs[1, 2])
sample = weekly.sample(min(2000, len(weekly)), random_state=42)
scatter_colors = sample['risk_tier'].map({
    'Low': GREEN, 'Moderate': GOLD, 'High': RED, 'Critical': '#8B0000'
})
ax5.scatter(sample['avg_stress'], sample['avg_sleep'],
            c=scatter_colors, alpha=0.35, s=18, linewidths=0)
ax5.set_title('Sleep vs Stress (colored by risk tier)')
ax5.set_xlabel('Avg Weekly Stress (1–10)')
ax5.set_ylabel('Avg Weekly Sleep (hours)')
patches = [mpatches.Patch(color=c, label=t) for t, c in RISK_COLORS.items()]
ax5.legend(handles=patches, fontsize=8, loc='upper right')

plt.suptitle('Burnout Risk Index (BRI) — Validation Dashboard', fontsize=15, y=1.01)
plt.savefig('../data/processed/plot_bri_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved BRI dashboard")


## 5. Frame as a Prediction Problem

> 💡 **Retrospective vs Predictive**
>
> So far the BRI is *retrospective* — it tells us how burnt out a student WAS
> this week, using this week's data. That's useful for dashboards but not 
> for interventions.
>
> For a real system, we want to predict: **"Will this student enter high-burnout 
> territory NEXT week?"** using only data from the CURRENT week.
>
> This is called a **lead-lag target** — the label (next week's burnout) is shifted 
> one period ahead of the features (this week's behavior). This ensures the model 
> has genuinely predictive power, not just a retrospective summary.


In [ ]:
weekly = weekly.sort_values(['student_id', 'week_num']).reset_index(drop=True)

weekly['next_BRI'] = weekly.groupby('student_id')['BRI'].shift(-1)

BURNOUT_THRESHOLD = 50
weekly['high_burnout_next_week'] = (weekly['next_BRI'] > BURNOUT_THRESHOLD).astype(int)

weekly_model = weekly.dropna(subset=['next_BRI']).copy()

print(f"Modeling dataset: {len(weekly_model):,} student-week observations")
print(f"\nTarget distribution:")
print(f"  Will be HIGH burnout next week (1): {weekly_model['high_burnout_next_week'].sum():,} "
      f"({weekly_model['high_burnout_next_week'].mean()*100:.1f}%)")
print(f"  Will NOT be high burnout (0)      : {(weekly_model['high_burnout_next_week']==0).sum():,} "
      f"({(1-weekly_model['high_burnout_next_week'].mean())*100:.1f}%)")
print(f"\n→ This is the class ratio we'll need to handle in modeling.")


## 6. Feature Engineering — Temporal & Behavioral Features

> 💡 **Temporal features are crucial for burnout**
>
> Burnout is path-dependent — it matters not just WHERE you are now, 
> but whether you're getting WORSE or recovering.
>
> We engineer:
> - **Trend features** (is burnout trending up or down?)
> - **Rolling averages** (3-week moving average for smoothing)
> - **Cumulative stress load** (total stress accumulated in the semester)
> - **Recovery indicators** (did BRI decrease from last week?)


In [ ]:
w = weekly_model.sort_values(['student_id', 'week_num']).copy()

w['bri_change']   = w.groupby('student_id')['BRI'].diff()
w['sleep_change'] = w.groupby('student_id')['avg_sleep'].diff()
w['stress_change']= w.groupby('student_id')['avg_stress'].diff()
w['mood_change']  = w.groupby('student_id')['avg_mood'].diff()

w['bri_rolling3'] = w.groupby('student_id')['BRI'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)

w['cumulative_stress'] = w.groupby('student_id')['avg_stress'].cumsum()

w['recovering'] = (w['bri_change'] < -3).astype(int)

def count_consecutive_high_stress(stress_series, threshold=6):
    result = []
    count = 0
    for val in stress_series:
        if val >= threshold:
            count += 1
        else:
            count = 0
        result.append(count)
    return result

w['consecutive_high_stress'] = w.groupby('student_id')['avg_stress'].transform(
    lambda x: count_consecutive_high_stress(x.values)
)

trend_cols = ['bri_change', 'sleep_change', 'stress_change', 'mood_change']
w[trend_cols] = w[trend_cols].fillna(0)

print("✓ Temporal & behavioral features engineered:")
new_features = ['bri_change', 'sleep_change', 'stress_change', 'mood_change',
                'bri_rolling3', 'cumulative_stress', 'recovering', 'consecutive_high_stress']
for f in new_features:
    print(f"  - {f}")
print(f"\nDataset shape: {w.shape}")


## 7. Build Feature Matrix & Train/Test Split

In [ ]:
FEATURES = [
    'avg_sleep', 'avg_stress', 'std_stress', 'avg_focus',
    'avg_mood', 'avg_caffeine', 'avg_activity', 'avg_social',
    'avg_sleep_debt', 'avg_stress_mood', 'avg_productivity',
    'avg_goal_rate', 'is_deadline_week', 'days_logged',
    'BRI', 'bri_rolling3',
    'bri_change', 'sleep_change', 'stress_change', 'mood_change',
    'cumulative_stress', 'recovering', 'consecutive_high_stress',
    'week_num',
]

TARGET = 'high_burnout_next_week'

df_model = w[FEATURES + [TARGET]].dropna()
X = df_model[FEATURES]
y = df_model[TARGET]

# Temporal split — train on earlier weeks, test on later weeks
# Using random split would leak future information = data leakage
train_mask = w.loc[df_model.index, 'week_num'] <= 14
test_mask  = ~train_mask

X_train = X[train_mask];  y_train = y[train_mask]
X_test  = X[test_mask];   y_test  = y[test_mask]

print(f"Temporal split (time-based, NOT random):")
print(f"  Train: weeks 1–14  → {len(X_train):,} rows  (positive rate: {y_train.mean()*100:.1f}%)")
print(f"  Test : weeks 15–18 → {len(X_test):,}  rows  (positive rate: {y_test.mean()*100:.1f}%)")
print(f"\n💡 Temporal split prevents future data leaking into training.")


## 8. Baseline — Logistic Regression

In [ ]:
baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

baseline.fit(X_train, y_train)
y_prob_base = baseline.predict_proba(X_test)[:, 1]
y_pred_base = baseline.predict(X_test)

print("BASELINE — Logistic Regression")
print("=" * 45)
print(f"Test ROC-AUC : {roc_auc_score(y_test, y_prob_base):.4f}")
print(f"Test F1      : {f1_score(y_test, y_pred_base, average='weighted'):.4f}")
print()
print(classification_report(y_test, y_pred_base, target_names=['Low burnout', 'High burnout']))


## 9. LightGBM Model

> 💡 **Why LightGBM instead of XGBoost here?**
>
> Both are gradient boosting frameworks, but LightGBM:
> - Uses "leaf-wise" tree growth (XGBoost uses "level-wise")
> - Is significantly faster on larger datasets
> - Handles class imbalance natively with `is_unbalance=True`
>
> Using a different algorithm across notebooks shows breadth.
> Notebook 03 = XGBoost. Notebook 04 = LightGBM.
> A recruiter reviewing your work sees you're not just copy-pasting the same model.
>
> Key parameters:
> - `num_leaves` — max leaves per tree (higher = more complex, more overfit risk)
> - `min_child_samples` — min samples in a leaf (prevents overfit)
> - `lambda_l1/l2` — regularization (L1 + L2 = ElasticNet style)


In [ ]:
lgbm_model = lgb.LGBMClassifier(
    n_estimators      = 500,
    learning_rate     = 0.03,
    num_leaves        = 31,
    max_depth         = -1,
    min_child_samples = 20,
    lambda_l1         = 0.1,
    lambda_l2         = 0.1,
    feature_fraction  = 0.8,
    bagging_fraction  = 0.8,
    bagging_freq      = 5,
    is_unbalance      = True,
    random_state      = 42,
    verbose           = -1,
    n_jobs            = -1,
)

lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(period=-1)]
)

y_prob_lgbm = lgbm_model.predict_proba(X_test)[:, 1]
y_pred_lgbm = lgbm_model.predict(X_test)

print("LightGBM Model")
print("=" * 45)
print(f"Best iteration : {lgbm_model.best_iteration_}")
print(f"Test ROC-AUC   : {roc_auc_score(y_test, y_prob_lgbm):.4f}")
print(f"Test F1        : {f1_score(y_test, y_pred_lgbm, average='weighted'):.4f}")
print()
print(classification_report(y_test, y_pred_lgbm, target_names=['Low burnout', 'High burnout']))


## 10. Model Comparison & Threshold Tuning

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
RocCurveDisplay.from_predictions(y_test, y_prob_base,
    name=f'Logistic Reg (AUC={roc_auc_score(y_test, y_prob_base):.3f})', ax=ax, color=BLUE)
RocCurveDisplay.from_predictions(y_test, y_prob_lgbm,
    name=f'LightGBM (AUC={roc_auc_score(y_test, y_prob_lgbm):.3f})', ax=ax, color=RED)
ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.4,label='Random')
ax.set_title('ROC Curve Comparison')
ax.legend(fontsize=9)

ax = axes[1]
thresholds = np.arange(0.1, 0.9, 0.02)
precisions, recalls, f1s = [], [], []

from sklearn.metrics import precision_score, recall_score
for t in thresholds:
    pred_t = (y_prob_lgbm >= t).astype(int)
    precisions.append(precision_score(y_test, pred_t, zero_division=0))
    recalls.append(recall_score(y_test, pred_t, zero_division=0))
    f1s.append(f1_score(y_test, pred_t, zero_division=0))

ax.plot(thresholds, precisions, label='Precision', color=BLUE,  linewidth=2)
ax.plot(thresholds, recalls,    label='Recall',    color=RED,   linewidth=2)
ax.plot(thresholds, f1s,        label='F1',        color=GREEN, linewidth=2)

best_t = thresholds[np.argmax(f1s)]
ax.axvline(best_t, color='gray', linestyle='--', linewidth=1.5, label=f'Best threshold = {best_t:.2f}')
ax.set_title('Threshold Analysis\n(precision vs recall tradeoff)')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.legend(fontsize=9)

print(f"Optimal threshold (max F1): {best_t:.2f}")
pred_opt = (y_prob_lgbm >= best_t).astype(int)
print(f"  Precision : {precision_score(y_test, pred_opt):.3f}")
print(f"  Recall    : {recall_score(y_test, pred_opt):.3f}")
print(f"  F1        : {f1_score(y_test, pred_opt):.3f}")

ax = axes[2]
cm = confusion_matrix(y_test, pred_opt)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Low burnout', 'High burnout'],
            yticklabels=['Low burnout', 'High burnout'],
            linewidths=0.5)
ax.set_title(f'Confusion Matrix\n(threshold = {best_t:.2f})')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('../data/processed/plot_burnout_model_eval.png', dpi=150, bbox_inches='tight')
plt.show()


## 11. SHAP Explainability — What Drives Burnout Risk?

> 💡 **SHAP for LightGBM**
>
> We use `shap.TreeExplainer` again — it works for all tree models
> (XGBoost, LightGBM, CatBoost, Random Forest, etc.).
> The interpretation is the same: positive SHAP = pushes toward high burnout.


In [ ]:
print("Computing SHAP values...")
explainer_b   = shap.TreeExplainer(lgbm_model)
shap_values_b = explainer_b.shap_values(X_test)

if isinstance(shap_values_b, list):
    shap_vals_b = shap_values_b[1]
else:
    shap_vals_b = shap_values_b

print(f"✓ SHAP values shape: {shap_vals_b.shape}")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

plt.sca(axes[0])
shap.summary_plot(shap_vals_b, X_test, feature_names=FEATURES,
                  plot_type='dot', max_display=14, show=False)
axes[0].set_title('SHAP Beeswarm — Burnout Risk Drivers\n'
                  'Red = high feature value, Blue = low feature value', fontsize=12)

plt.sca(axes[1])
shap.summary_plot(shap_vals_b, X_test, feature_names=FEATURES,
                  plot_type='bar', max_display=14, show=False)
axes[1].set_title('Mean |SHAP| — Feature Importance for Burnout', fontsize=12)

plt.tight_layout()
plt.savefig('../data/processed/plot_burnout_shap.png', dpi=150, bbox_inches='tight')
plt.show()


## 12. Individual Student Burnout Trajectories

> 💡 **Why individual trajectories matter**
>
> Aggregate stats tell us about the population. Trajectories tell us about 
> individuals — which is what a real burnout monitoring system needs.
> This chart is the kind of visualization a product manager or therapist 
> would actually use.


In [ ]:
student_avg_bri = w.groupby('student_id')['BRI'].mean().sort_values()

low_risk_students  = student_avg_bri.head(100).sample(2, random_state=1).index.tolist()
high_risk_students = student_avg_bri.tail(100).sample(2, random_state=2).index.tolist()
mid_risk_students  = student_avg_bri.iloc[200:300].sample(2, random_state=3).index.tolist()

selected_students = low_risk_students + mid_risk_students + high_risk_students
labels      = ['Low risk']*2 + ['Medium risk']*2 + ['High risk']*2
colors_traj = [GREEN, GREEN, GOLD, GOLD, RED, RED]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Individual Burnout Risk Trajectories Across Semester', fontsize=14, y=1.01)

for ax, stu_id, label, color in zip(axes.flatten(), selected_students, labels, colors_traj):
    stu_data = w[w['student_id'] == stu_id].sort_values('week_num')

    ax.fill_between(stu_data['week_num'], stu_data['BRI'], alpha=0.2, color=color)
    ax.plot(stu_data['week_num'], stu_data['BRI'], color=color, linewidth=2.2, marker='o', markersize=4)

    ax.axhspan(0,  25,  alpha=0.05, color=GREEN)
    ax.axhspan(25, 50,  alpha=0.05, color=GOLD)
    ax.axhspan(50, 75,  alpha=0.05, color=RED)
    ax.axhspan(75, 100, alpha=0.05, color='darkred')

    for dw in stu_data[stu_data['is_deadline_week']==1]['week_num']:
        ax.axvline(dw, color='gray', linestyle=':', linewidth=1, alpha=0.7)

    stu_idx = X_test.index.intersection(w[w['student_id']==stu_id].index)
    if len(stu_idx) > 0:
        stu_pred_weeks = w.loc[stu_idx, 'week_num']
        stu_pred_probs = lgbm_model.predict_proba(X_test.loc[stu_idx])[:,1] * 100
        ax2 = ax.twinx()
        ax2.plot(stu_pred_weeks, stu_pred_probs,
                 linestyle='--', color='purple', linewidth=1.5, alpha=0.7, label='Pred. risk %')
        ax2.set_ylim(0, 100)
        ax2.set_ylabel('Predicted risk %', color='purple', fontsize=8)
        ax2.tick_params(axis='y', labelcolor='purple', labelsize=8)

    avg_bri = stu_data['BRI'].mean()
    ax.set_title(f'{stu_id} — {label}\nAvg BRI: {avg_bri:.1f}', fontsize=10)
    ax.set_xlabel('Week')
    ax.set_ylabel('Burnout Risk Index')
    ax.set_ylim(0, 100)
    ax.axhline(50, color='gray', linestyle='--', linewidth=1, alpha=0.5)

plt.tight_layout()
plt.savefig('../data/processed/plot_burnout_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()


## 13. Module 3 Summary — Burnout Risk Findings

In [ ]:
print("=" * 60)
print("MODULE 3 — BURNOUT RISK SCORING: KEY FINDINGS")
print("=" * 60)

base_auc = roc_auc_score(y_test, y_prob_base)
lgbm_auc = roc_auc_score(y_test, y_prob_lgbm)
lift = (lgbm_auc - base_auc) / base_auc * 100

print(f"\n1. Burnout Risk Index (BRI):")
print(f"   Scale: 0–100  |  Threshold for 'High Risk': 50")
tier_dist = weekly['risk_tier'].value_counts(normalize=True).sort_index() * 100
for tier, pct in tier_dist.items():
    print(f"   {tier:<12}: {pct:.1f}% of student-weeks")

print(f"\n2. Predictive Model Performance (next-week burnout):")
print(f"   Logistic Regression AUC : {base_auc:.4f}")
print(f"   LightGBM AUC            : {lgbm_auc:.4f}  (+{lift:.1f}% lift)")
print(f"   Optimal threshold       : {best_t:.2f}")

print(f"\n3. Top burnout risk drivers (SHAP):")
mean_shap_b = pd.Series(np.abs(shap_vals_b).mean(axis=0), index=FEATURES)
for feat, val in mean_shap_b.sort_values(ascending=False).head(5).items():
    print(f"   {feat:<30} mean |SHAP| = {val:.4f}")

print(f"\n4. Deadline weeks effect:")
deadline_bri = w[w['is_deadline_week']==1]['BRI'].mean()
normal_bri   = w[w['is_deadline_week']==0]['BRI'].mean()
print(f"   Avg BRI in deadline weeks  : {deadline_bri:.1f}")
print(f"   Avg BRI in normal weeks    : {normal_bri:.1f}")
print(f"   Increase during deadlines  : +{deadline_bri - normal_bri:.1f} points")

print(f"\n→ NEXT: Notebook 05 — Productivity Forecasting (Prophet)")
print(f"  Forecasting next 7 days of peak productivity windows per student")
